In [1]:
%pip install "moss>=1.0.0" dspy -q


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import asyncio
import dspy
from dspy.dsp.utils import dotdict
from dspy.primitives.prediction import Prediction
from dspy.retrievers.retrieve import Retrieve
from moss import DocumentInfo, MossClient, QueryOptions
import nest_asyncio
import os
from dotenv import load_dotenv

load_dotenv()
nest_asyncio.apply()

def run_async(coro):
    """Helper to run async coroutines in a notebook environment."""
    try:
        loop = asyncio.get_event_loop()
    except RuntimeError:
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
    
    if loop.is_running():
        return loop.run_until_complete(coro)
    else:
        return asyncio.run(coro)

class MossRM(Retrieve):
    """A retrieval module that uses Moss to return the top passages for a given query.

    Args:
        index_name (str): The name of the Moss index.
        moss_client (MossClient): An instance of the Moss client.
        k (int, optional): The default number of top passages to retrieve. Default to 3.

    Examples:
        Below is a code snippet that shows how to use Moss as the default retriever:
        ```python
        from moss import MossClient
        import dspy

        moss_client = MossClient("your-project-id", "your-project-key")
        retriever_model = MossRM("my_index_name", moss_client=moss_client)
        dspy.configure(rm=retriever_model)

        retrieve = dspy.Retrieve(k=1)
        topK_passages = retrieve("what are the stages in planning, sanctioning and execution of public works").passages
        ```
    """

    def __init__(
        self,
        index_name: str,
        moss_client: MossClient,
        k: int = 3,
        alpha: float = 0.5,
    ):
        self._index_name = index_name
        self._moss_client = moss_client
        self._alpha = alpha

        super().__init__(k=k)

    def forward(self, query_or_queries: str | list[str], k: int | None = None, **kwargs) -> Prediction:
        """Search with Moss for self.k top passages for query or queries.

        Args:
            query_or_queries (Union[str, list[str]]): The query or queries to search for.
            k (Optional[int]): The number of top passages to retrieve. Defaults to self.k.
            kwargs : Additional arguments for Moss client.

        Returns:
            dspy.Prediction: An object containing the retrieved passages.
        """
        k = k if k is not None else self.k
        queries = [query_or_queries] if isinstance(query_or_queries, str) else query_or_queries
        queries = [q for q in queries if q]
        passages = []

        for query in queries:
            options = QueryOptions(top_k=k, alpha=self._alpha, **kwargs)
            # Since MossClient methods are async, we use asyncio.run to call them synchronously.
            # This assumes the loop is not already running, which is typical for DSPy RM calls.
            result = asyncio.run(self._moss_client.query(self._index_name, query, options=options))

            for doc in result.docs:
                passages.append(
                    dotdict({"long_text": doc.text, "id": doc.id, "metadata": doc.metadata, "score": doc.score})
                )

        return passages

    def get_objects(self, num_samples: int = 5) -> list[dict]:
        """Get objects from Moss."""
        # Note: Moss's get_docs might return all docs or have limits.
        # Here we attempt to fetch and return up to num_samples.
        result = asyncio.run(self._moss_client.get_docs(self._index_name))
        # result is likely a list of DocumentInfo or similar
        objects = []
        for i, doc in enumerate(result):
            if i >= num_samples:
                break
            objects.append({"id": doc.id, "text": doc.text, "metadata": doc.metadata})
        return objects

    def insert(self, new_object_properties: dict | list[dict]):
        """Insert one or more objects into Moss."""
        if isinstance(new_object_properties, dict):
            new_object_properties = [new_object_properties]

        docs = [DocumentInfo(**props) for props in new_object_properties]
        asyncio.run(self._moss_client.add_docs(self._index_name, docs))

In [24]:
import requests
import os
import asyncio
from moss import MossClient, DocumentInfo
# 2. Initialize MossClient (Replace with your actual keys)
MOSS_PROJECT_ID = os.getenv("MOSS_PROJECT_ID", os.getenv("MOSS_PROJECT_ID"))
MOSS_PROJECT_KEY = os.getenv("MOSS_PROJECT_KEY", os.getenv("MOSS_PROJECT_KEY"))

client = MossClient(MOSS_PROJECT_ID, MOSS_PROJECT_KEY)
INDEX_NAME = "moss-dspy"

llm = dspy.LM(model="gpt-4.1-mini",api_key=os.getenv("OPENAI_API_KEY"))
# retriever_model = MossRM(INDEX_NAME, moss_client=client) ways to use Moss as retriever
# dspy.settings.configure(lm=llm, rm=retriever_model)
dspy.configure(lm=llm)

In [25]:
from typing import List

documents: List[DocumentInfo] = [
    DocumentInfo(
        id="faq-shipping-001",
        text=(
            "How can I change my shipping address? Contact our customer service team at support@store.com "
            "or call 1-800-555-0100 before your order is processed. Once the order has shipped, address "
            "changes are no longer possible. You can also update your default shipping address anytime "
            "from your account settings under 'Addresses'."
        ),
        metadata={"category": "shipping", "type": "faq"},
    ),
    DocumentInfo(
        id="faq-shipping-002",
        text=(
            "What are the estimated delivery times? Standard shipping takes 5–7 business days. Expedited "
            "shipping arrives in 2–3 business days. Overnight shipping guarantees next-day delivery if "
            "ordered before 2 PM EST. International orders typically take 10–21 business days depending "
            "on the destination country and local customs processing."
        ),
        metadata={"category": "shipping", "type": "faq"},
    ),
    DocumentInfo(
        id="faq-returns-001",
        text=(
            "What is the return policy? Items can be returned within 30 days of delivery for a full refund "
            "to your original payment method. Products must be unused and in original packaging. To start "
            "a return, visit your order history, select the item, and click 'Return Item'. A prepaid "
            "shipping label will be emailed to you within 24 hours."
        ),
        metadata={"category": "returns", "type": "faq"},
    ),
    DocumentInfo(
        id="faq-returns-002",
        text=(
            "Can I exchange an item instead of returning it? Yes, exchanges are available for size or "
            "color changes on most clothing and footwear. To request an exchange, contact customer support "
            "within 30 days of delivery. If the desired variant is out of stock, we will issue a store "
            "credit valid for 12 months."
        ),
        metadata={"category": "returns", "type": "faq"},
    ),
    DocumentInfo(
        id="faq-payment-001",
        text=(
            "Which payment methods are accepted? We accept Visa, Mastercard, American Express, Discover, "
            "PayPal, Apple Pay, and Google Pay. We also offer buy-now-pay-later through Affirm with 0% "
            "APR for qualifying orders over $50. Gift cards issued by our store can be applied at checkout "
            "and combined with any other payment method."
        ),
        metadata={"category": "payment", "type": "faq"},
    ),
    DocumentInfo(
        id="faq-account-001",
        text=(
            "How do I reset my password? On the login page, click 'Forgot Password' and enter your "
            "registered email address. You will receive a reset link within a few minutes — check your "
            "spam folder if it does not appear. The link expires after 60 minutes. If you no longer have "
            "access to that email address, contact support to verify your identity and update your account."
        ),
        metadata={"category": "account", "type": "faq"},
    ),
    DocumentInfo(
        id="faq-orders-001",
        text=(
            "How do I track my order? Once your order ships, a tracking number will be emailed to you. "
            "You can also find it in your account under 'Order History'. Click the tracking number to be "
            "redirected to the carrier's website for real-time updates. If tracking has not updated in "
            "more than 5 business days, please contact our support team."
        ),
        metadata={"category": "orders", "type": "faq"},
    ),
    DocumentInfo(
        id="faq-orders-002",
        text=(
            "Can I cancel or modify my order after placing it? Orders can be cancelled or modified within "
            "1 hour of placement by visiting 'Order History' and selecting 'Cancel Order'. After that "
            "window, the order enters fulfillment and cannot be changed. If it has already shipped, "
            "please initiate a return once the package arrives."
        ),
        metadata={"category": "orders", "type": "faq"},
    ),
]

await client.create_index(INDEX_NAME,documents)

In [27]:
# Create a ReAct agent using Moss as a tool
def moss_search(query: str):
    """Searches the Moss knowledge base for relevant passages."""
    
    # 2. Configure query options
    options = QueryOptions(top_k=TOP_K, alpha=0)
    # 3. Use the run_async helper to call Moss from a sync function
    # (Results are automatically returned as a list of documents)
    results = run_async(client.query(INDEX_NAME, query, options=options))
    # You can process multiple topK results as well
    if results.docs:
        print(f"[Tool DEBUG] Found {len(results.docs)} results for: {query}")
        print(f"[Tool DEBUG] First snippet: {results.docs[0].text}...")
    
    if not results.docs:
        return "No relevant info found."
        
    return "\n".join([f"- {doc.text}" for doc in results.docs])


TOP_K = 10
await client.load_index(INDEX_NAME)

# Initialize the ReAct agent with the Moss search tool
react_agent = dspy.ReAct(
    signature="question -> answer",
    tools=[moss_search],
    max_iters=5
)

# Example usage
question = "Which payment modes are accepted?"
result = react_agent(question=question)

print(f"Answer: {result.answer}")
print("\nTool calls made (Trajectory):")
for step in result.trajectory:
    print(step)

[Tool DEBUG] Found 8 results for: payment modes accepted
[Tool DEBUG] First snippet: Which payment methods are accepted? We accept Visa, Mastercard, American Express, Discover, PayPal, Apple Pay, and Google Pay. We also offer buy-now-pay-later through Affirm with 0% APR for qualifying orders over $50. Gift cards issued by our store can be applied at checkout and combined with any other payment method....
Answer: The accepted payment modes are Visa, Mastercard, American Express, Discover, PayPal, Apple Pay, Google Pay, buy-now-pay-later through Affirm (with 0% APR for qualifying orders over $50), and store-issued gift cards which can be combined with any other payment method.

Tool calls made (Trajectory):
thought_0
tool_name_0
tool_args_0
observation_0
thought_1
tool_name_1
tool_args_1
observation_1
